# Disease Prediction from Medical Data
**Heart disease · Diabetes · Breast cancer** with SVM, Logistic Regression, Random Forest and XGBoost.

This notebook is fully self-contained: it creates every project file itself and downloads the datasets automatically. No uploads needed. Just click **Runtime → Run all** (about 4-5 minutes).

> Research / educational use only. Not a medical diagnosis tool.

## 1. Create the project files

In [ ]:
import os
# Colab: /content  |  VS Code / local: your home folder
BASE = '/content' if os.path.isdir('/content') else os.path.expanduser('~')
os.chdir(BASE)
for d in ['src/disease_prediction', 'tests', 'data/raw']:
    os.makedirs(f'disease-prediction/{d}', exist_ok=True)
print('Project folder:', os.path.join(BASE, 'disease-prediction'))

In [ ]:
%%writefile disease-prediction/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "disease-prediction"
version = "1.0.0"
description = "Disease prediction from structured medical data (heart disease, diabetes, breast cancer)"
requires-python = ">=3.9"
dependencies = [
    "numpy", "pandas", "scikit-learn", "xgboost", "matplotlib", "joblib",
]

[project.optional-dependencies]
dev = ["pytest"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]


In [ ]:
%%writefile disease-prediction/src/disease_prediction/__init__.py
"""Disease prediction from structured medical data."""
__version__ = "1.0.0"


In [ ]:
%%writefile disease-prediction/src/disease_prediction/config.py
"""Central configuration: paths and experiment constants."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = ROOT / "data" / "raw"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
N_BOOTSTRAP = 1000          # resamples for the ROC-AUC confidence interval
SELECTION_METRIC = "roc_auc"  # used to tune and to pick the best model (CV only, never the test set)

for _d in (DATA_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile disease-prediction/src/disease_prediction/data.py
"""Dataset loading.

Every loader returns a DataFrame whose ``target`` column is binary with
**1 = disease present** and 0 = disease absent, regardless of how the original
source coded it. Raw files are cached in ``data/raw`` and re-downloaded if missing.
"""
from __future__ import annotations

import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

from .config import DATA_DIR

log = logging.getLogger(__name__)
TARGET = "target"


@dataclass(frozen=True)
class DatasetSpec:
    key: str
    title: str
    description: str
    categorical: tuple = field(default_factory=tuple)  # one-hot encoded; everything else is numeric


def _ensure_file(path: Path, url: str) -> Path:
    if not path.exists():
        log.info("Downloading %s", url)
        urllib.request.urlretrieve(url, path)
    return path


def load_heart() -> pd.DataFrame:
    """UCI Heart Disease (Cleveland), 303 patients, 13 clinical features."""
    path = _ensure_file(
        DATA_DIR / "heart_disease_cleveland.csv",
        "https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv",
    )
    df = pd.read_csv(path, encoding="utf-8-sig")
    # This widely-mirrored copy codes 1 = *no* disease (verified: target==1 patients have higher
    # max heart rate and less angina / ST depression). Flip so that 1 = disease present.
    df[TARGET] = 1 - df[TARGET]
    return df.drop_duplicates().reset_index(drop=True)


def load_diabetes() -> pd.DataFrame:
    """Pima Indians Diabetes (UCI / NIDDK), 768 patients, 8 features."""
    path = _ensure_file(
        DATA_DIR / "pima_diabetes.csv",
        "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv",
    )
    cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
            "BMI", "DiabetesPedigreeFunction", "Age", TARGET]
    df = pd.read_csv(path, header=None, names=cols)
    # A value of 0 is physiologically impossible for these measurements -> it encodes "missing".
    df[["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]] = (
        df[["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]].replace(0, np.nan)
    )
    return df.drop_duplicates().reset_index(drop=True)


def load_breast_cancer_df() -> pd.DataFrame:
    """Wisconsin Diagnostic Breast Cancer (UCI), 569 samples, 30 cell-nucleus features."""
    bunch = load_breast_cancer(as_frame=True)
    df = bunch.frame.copy()
    df.columns = [c.replace(" ", "_") for c in df.columns]
    # sklearn codes malignant = 0 / benign = 1. Flip so 1 = malignant (disease present).
    df[TARGET] = 1 - df[TARGET]
    return df.drop_duplicates().reset_index(drop=True)


REGISTRY = {
    "heart": (DatasetSpec(
        "heart", "Heart Disease (Cleveland)",
        "Predict presence of coronary heart disease from clinical measurements.",
        categorical=("cp", "restecg", "slope", "thal")), load_heart),
    "diabetes": (DatasetSpec(
        "diabetes", "Diabetes (Pima Indians)",
        "Predict onset of diabetes from diagnostic measurements."), load_diabetes),
    "breast_cancer": (DatasetSpec(
        "breast_cancer", "Breast Cancer (Wisconsin Diagnostic)",
        "Classify a tumour as malignant vs. benign from cell-nucleus features."), load_breast_cancer_df),
}


def load_dataset(key: str):
    """Return (spec, X, y) for a dataset key."""
    if key not in REGISTRY:
        raise KeyError(f"Unknown dataset '{key}'. Choose from {list(REGISTRY)}")
    spec, loader = REGISTRY[key]
    df = loader()
    return spec, df.drop(columns=[TARGET]), df[TARGET].astype(int)


In [ ]:
%%writefile disease-prediction/src/disease_prediction/preprocessing.py
"""Leakage-safe preprocessing. Fitted only inside the CV / training folds via sklearn Pipelines."""
from __future__ import annotations

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from .data import DatasetSpec


def build_preprocessor(spec: DatasetSpec, X: pd.DataFrame) -> ColumnTransformer:
    cat = [c for c in spec.categorical if c in X.columns]
    num = [c for c in X.columns if c not in cat]

    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    transformers = [("num", numeric, num)]
    if cat:
        categorical = Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("cat", categorical, cat))
    return ColumnTransformer(transformers)


In [ ]:
%%writefile disease-prediction/src/disease_prediction/models.py
"""Model zoo: SVM, Logistic Regression, Random Forest, XGBoost + their tuning grids."""
from __future__ import annotations

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

from .config import SEED


def get_model_zoo(pos_weight: float = 1.0) -> dict:
    """Return {name: (estimator, param_grid)}. Grid keys target the pipeline step ``clf``.

    ``pos_weight`` = n_negative / n_positive, used by XGBoost to handle class imbalance
    (the other models use ``class_weight='balanced'``).
    """
    return {
        "Logistic Regression": (
            LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED),
            {"clf__C": [0.01, 0.1, 1, 10]},
        ),
        "SVM": (
            SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
            {"clf__C": [0.5, 1, 5, 10], "clf__gamma": ["scale", 0.01, 0.1]},
        ),
        "Random Forest": (
            RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
            {"clf__n_estimators": [200, 400],
             "clf__max_depth": [4, 8, None],
             "clf__min_samples_leaf": [1, 3]},
        ),
        "XGBoost": (
            XGBClassifier(eval_metric="logloss", subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=pos_weight, n_jobs=-1, random_state=SEED, verbosity=0),
            {"clf__n_estimators": [150, 300],
             "clf__max_depth": [2, 3, 4],
             "clf__learning_rate": [0.03, 0.1]},
        ),
    }


In [ ]:
%%writefile disease-prediction/src/disease_prediction/evaluate.py
"""Metrics, confidence intervals and plots."""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)

from .config import N_BOOTSTRAP, SEED


def positive_scores(model, X) -> np.ndarray:
    """Probability of the disease class."""
    return model.predict_proba(X)[:, 1]


def compute_metrics(y_true, y_pred, y_score) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall (Sensitivity)": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
        "F1-Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "PR-AUC": average_precision_score(y_true, y_score),
    }


def bootstrap_auc_ci(y_true, y_score, n: int = N_BOOTSTRAP, alpha: float = 0.05):
    """Percentile bootstrap CI for ROC-AUC (test sets here are small, so report uncertainty)."""
    rng = np.random.default_rng(SEED)
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    aucs = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    return float(np.percentile(aucs, 100 * alpha / 2)), float(np.percentile(aucs, 100 * (1 - alpha / 2)))


# ------------------------------------------------------------------ plots
def plot_roc(scores: dict, y_test, title: str, path):
    plt.figure(figsize=(6.5, 5.5))
    for name, s in scores.items():
        fpr, tpr, _ = roc_curve(y_test, s)
        plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, s):.3f})")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
    plt.xlabel("False positive rate (1 - specificity)")
    plt.ylabel("True positive rate (sensitivity)")
    plt.title(f"ROC curves - {title}")
    plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(path, dpi=150); plt.close()


def plot_confusion(model, X_test, y_test, title: str, path):
    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["No disease", "Disease"], cmap="Blues", ax=ax)
    ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def permutation_importances(model, X_test, y_test) -> pd.Series:
    """Model-agnostic importance measured on the original (un-encoded) features."""
    r = permutation_importance(model, X_test, y_test, scoring="roc_auc",
                               n_repeats=30, random_state=SEED, n_jobs=-1)
    return pd.Series(r.importances_mean, index=X_test.columns).sort_values(ascending=False)


def plot_importance(imp: pd.Series, title: str, path, top: int = 10):
    imp = imp.head(top)[::-1]
    plt.figure(figsize=(6.5, 4.5))
    imp.plot.barh(color="#2563eb")
    plt.xlabel("Drop in ROC-AUC when feature is shuffled")
    plt.title(title, fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


In [ ]:
%%writefile disease-prediction/src/disease_prediction/train.py
"""Train, tune, evaluate and persist models.

    python -m disease_prediction.train --dataset all
    python -m disease_prediction.train --dataset heart
"""
from __future__ import annotations

import argparse
import logging
import time

import joblib
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

from . import __version__
from .config import (CV_FOLDS, FIGURES_DIR, MODELS_DIR, REPORTS_DIR, SEED,
                     SELECTION_METRIC, TEST_SIZE)
from .data import REGISTRY, load_dataset
from .evaluate import (bootstrap_auc_ci, compute_metrics, permutation_importances, plot_confusion,
                       plot_importance, plot_roc, positive_scores)
from .models import get_model_zoo
from .preprocessing import build_preprocessor

log = logging.getLogger("train")


def run_dataset(key: str) -> pd.DataFrame:
    spec, X, y = load_dataset(key)
    log.info("=== %s | %d rows, %d features, %.1f%% positive ===",
             spec.title, len(X), X.shape[1], 100 * y.mean())

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
    cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    rows, fitted, scores = [], {}, {}
    for name, (estimator, grid) in get_model_zoo(pos_weight).items():
        t0 = time.time()
        pipe = Pipeline([("prep", build_preprocessor(spec, X_train)), ("clf", estimator)])
        search = GridSearchCV(pipe, grid, scoring=SELECTION_METRIC, cv=cv, n_jobs=-1, refit=True)
        search.fit(X_train, y_train)
        model = search.best_estimator_

        y_score = positive_scores(model, X_test)
        y_pred = (y_score >= 0.5).astype(int)
        m = compute_metrics(y_test, y_pred, y_score)
        lo, hi = bootstrap_auc_ci(y_test, y_score)
        rows.append({"Dataset": spec.title, "Model": name,
                     "CV ROC-AUC": search.best_score_, **m,
                     "ROC-AUC 95% CI": f"[{lo:.3f}, {hi:.3f}]",
                     "Best params": {k.replace("clf__", ""): v for k, v in search.best_params_.items()}})
        fitted[name], scores[name] = model, y_score
        log.info("%-20s CV-AUC %.3f | test AUC %.3f  Recall %.3f  F1 %.3f  (%.1fs)",
                 name, search.best_score_, m["ROC-AUC"], m["Recall (Sensitivity)"], m["F1-Score"],
                 time.time() - t0)

    results = pd.DataFrame(rows)

    # Model selection uses CROSS-VALIDATION on the training split only, never the test set.
    best_name = results.loc[results["CV ROC-AUC"].idxmax(), "Model"]
    best = fitted[best_name]
    results["Selected"] = results["Model"].eq(best_name)
    log.info("Selected model (highest CV ROC-AUC): %s", best_name)

    # Artifacts
    plot_roc(scores, y_test, spec.title, FIGURES_DIR / f"{key}_roc.png")
    plot_confusion(best, X_test, y_test, f"{spec.title}\n{best_name} (threshold 0.5)",
                   FIGURES_DIR / f"{key}_confusion_matrix.png")
    imp = permutation_importances(best, X_test, y_test)
    plot_importance(imp, f"{spec.title} - top features ({best_name})",
                    FIGURES_DIR / f"{key}_feature_importance.png")
    results.to_csv(REPORTS_DIR / f"{key}_metrics.csv", index=False)
    joblib.dump({"pipeline": best, "model_name": best_name, "dataset": key,
                 "features": list(X.columns), "version": __version__,
                 "test_roc_auc": float(results.loc[results.Selected, "ROC-AUC"].iloc[0])},
                MODELS_DIR / f"{key}_best.joblib")
    return results


def write_summary(all_results: pd.DataFrame):
    cols = ["Model", "CV ROC-AUC", "Accuracy", "Precision", "Recall (Sensitivity)",
            "Specificity", "F1-Score", "ROC-AUC", "ROC-AUC 95% CI"]
    lines = ["# Results summary\n",
             "Hold-out test metrics (20% stratified split, threshold 0.5). "
             "`CV ROC-AUC` is the 5-fold score on the training split used for model selection (**bold** = selected).\n"]
    for ds, g in all_results.groupby("Dataset", sort=False):
        lines += [f"\n## {ds}\n", "| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
        for _, r in g.iterrows():
            cells = [f"{r[c]:.3f}" if isinstance(r[c], float) else str(r[c]) for c in cols]
            if r["Selected"]:
                cells = [f"**{c}**" for c in cells]
            lines.append("| " + " | ".join(cells) + " |")
    (REPORTS_DIR / "results_summary.md").write_text("\n".join(lines) + "\n")
    all_results.drop(columns=["Best params"]).to_csv(REPORTS_DIR / "results_summary.csv", index=False)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    ap.add_argument("--dataset", default="all", choices=["all", *REGISTRY])
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")

    keys = list(REGISTRY) if args.dataset == "all" else [args.dataset]
    all_results = pd.concat([run_dataset(k) for k in keys], ignore_index=True)
    write_summary(all_results)

    show = all_results[all_results.Selected][
        ["Dataset", "Model", "CV ROC-AUC", "Accuracy", "Recall (Sensitivity)", "F1-Score", "ROC-AUC"]]
    print("\nSelected models\n" + show.round(3).to_string(index=False))
    print(f"\nReports -> {REPORTS_DIR}\nModels  -> {MODELS_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile disease-prediction/src/disease_prediction/predict.py
"""Score new patients with a trained model.

    python -m disease_prediction.predict --dataset heart --input examples/heart_patient.json
Input: a JSON object (one patient) or a list of objects, with the dataset's feature columns.
"""
from __future__ import annotations

import argparse
import json

import joblib
import pandas as pd

from .config import MODELS_DIR


def risk_band(p: float) -> str:
    return "LOW" if p < 0.33 else "MODERATE" if p < 0.66 else "HIGH"


def predict(dataset: str, records: list[dict]) -> pd.DataFrame:
    path = MODELS_DIR / f"{dataset}_best.joblib"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - run `python -m disease_prediction.train` first.")
    bundle = joblib.load(path)
    X = pd.DataFrame(records)
    missing = set(bundle["features"]) - set(X.columns)
    if missing:
        raise ValueError(f"Missing required features: {sorted(missing)}")
    p = bundle["pipeline"].predict_proba(X[bundle["features"]])[:, 1]
    return pd.DataFrame({"disease_probability": p.round(3), "risk_band": [risk_band(v) for v in p]})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True, choices=["heart", "diabetes", "breast_cancer"])
    ap.add_argument("--input", required=True, help="Path to JSON file")
    args = ap.parse_args()
    data = json.load(open(args.input))
    print(predict(args.dataset, data if isinstance(data, list) else [data]).to_string(index=False))
    print("\nFor research/education only - not a medical diagnosis.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile disease-prediction/tests/test_pipeline.py
import numpy as np
import pytest
from sklearn.pipeline import Pipeline

from disease_prediction.data import REGISTRY, load_dataset
from disease_prediction.evaluate import bootstrap_auc_ci, compute_metrics
from disease_prediction.models import get_model_zoo
from disease_prediction.preprocessing import build_preprocessor


@pytest.mark.parametrize("key", list(REGISTRY))
def test_loader_contract(key):
    spec, X, y = load_dataset(key)
    assert len(X) == len(y) > 250
    assert set(y.unique()) == {0, 1}
    assert "target" not in X.columns


def test_target_polarity_is_disease_positive():
    # sanity checks that 1 really means "disease" after our flips
    _, X, y = load_dataset("heart")
    assert X.loc[y == 1, "thalach"].mean() < X.loc[y == 0, "thalach"].mean()
    _, X, y = load_dataset("breast_cancer")
    assert X.loc[y == 1, "mean_radius"].mean() > X.loc[y == 0, "mean_radius"].mean()


def test_diabetes_impossible_zeros_become_nan():
    _, X, _ = load_dataset("diabetes")
    assert X["Glucose"].isna().sum() > 0
    assert (X["Glucose"].dropna() > 0).all()


@pytest.mark.parametrize("key", list(REGISTRY))
@pytest.mark.parametrize("name", ["Logistic Regression", "SVM", "Random Forest", "XGBoost"])
def test_every_model_fits_and_outputs_probabilities(key, name):
    spec, X, y = load_dataset(key)
    est, _ = get_model_zoo()[name]
    pipe = Pipeline([("prep", build_preprocessor(spec, X)), ("clf", est)]).fit(X, y)
    p = pipe.predict_proba(X.head(20))[:, 1]
    assert p.shape == (20,) and ((p >= 0) & (p <= 1)).all()


def test_metrics_and_ci():
    y = np.array([0, 0, 1, 1, 1, 0, 1, 0])
    s = np.array([.1, .2, .9, .8, .6, .4, .7, .3])
    m = compute_metrics(y, (s >= .5).astype(int), s)
    assert m["ROC-AUC"] == 1.0 and m["Recall (Sensitivity)"] == 1.0 and m["Specificity"] == 1.0
    lo, hi = bootstrap_auc_ci(y, s, n=200)
    assert 0 <= lo <= hi <= 1


## 2. Install the package

In [ ]:
import os, sys
os.chdir(os.path.join(BASE, 'disease-prediction'))
!pip install -q -U "xgboost>=2.1.4"
!pip install -q -e .
# An editable install is not visible to an already-running notebook until restart,
# so add the source folder to the import path explicitly.
sys.path.insert(0, os.path.abspath('src'))
import disease_prediction
print('disease_prediction', disease_prediction.__version__, 'ready')

## 3. (Optional) Run unit tests

In [ ]:
!pip install -q pytest
!python -m pytest -q

## 4. Train, tune and evaluate all models
Downloads the Heart and Diabetes data on first run; Breast Cancer ships with scikit-learn.

In [ ]:
!python -m disease_prediction.train --dataset all 2>&1 | grep -vi warning

## 5. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

res = pd.read_csv('reports/results_summary.csv')
display(res[res.Selected].round(3))          # selected model per dataset
display(res.drop(columns='Selected').round(3))  # all models

In [ ]:
for ds in ['heart', 'diabetes', 'breast_cancer']:
    for kind in ['roc', 'confusion_matrix', 'feature_importance']:
        display(Image(f'reports/figures/{ds}_{kind}.png', width=480))

## 6. Predict for a new patient

In [ ]:
from disease_prediction.predict import predict

heart_patient = {"age": 58, "sex": 1, "cp": 0, "trestbps": 140, "chol": 270, "fbs": 0, "restecg": 0,
                 "thalach": 115, "exang": 1, "oldpeak": 2.5, "slope": 1, "ca": 2, "thal": 3}
diabetes_patient = {"Pregnancies": 3, "Glucose": 155, "BloodPressure": 78, "SkinThickness": 32, "Insulin": 180,
                    "BMI": 36.5, "DiabetesPedigreeFunction": 0.62, "Age": 45}

print('Heart   :'); display(predict('heart', [heart_patient]))
print('Diabetes:'); display(predict('diabetes', [diabetes_patient]))